# **1: Settings**


constants and paths, plus an optional decoder install that fails harmlessly if unneeded. PRIOR is the average training prevalence, used if a study ever fails.

In [ ]:
import os, glob, time, subprocess, sys, numpy as np, pandas as pd, torch, torch.nn as nn, pydicom, cv2, timm
ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
S, K = 192, 16
PLANES = ["Sagittal", "Coronal", "Axial"]
SPLIT = "test"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
PRIOR = np.array([.142, .155, .330, .174, .234, .169, .317, .445, .151, .271, .220, .068], dtype=np.float32)
dev = "cuda" if torch.cuda.is_available() else "cpu"
w = glob.glob("/kaggle/input/datasets/*/rsna-knee-pseudo-labels/wheels")
if w:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--find-links", w[0],
                        "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], capture_output=True, text=True)
    print("optional decoder install return code:", r.returncode)
print("device:", dev)

# **2: Preprocessing functions**

 the same preprocessing as in training, so the model sees identical inputs. 

In [ ]:
def pick_series(g):
    picks = []
    for plane in PLANES:
        c = g[g.Anatomical_Plane == plane]
        f = c[c.Fluid_Sensitive == 1]
        use = f if len(f) else c
        picks.append(use.SeriesInstanceUID.iloc[0] if len(use) else None)
    return picks

def slice_key(h):
    try:
        ipp = np.array(h.ImagePositionPatient, float); iop = np.array(h.ImageOrientationPatient, float)
        return float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
    except Exception:
        pass
    try:
        return float(h.InstanceNumber)
    except Exception:
        return 0.0

def load_series(study, series):
    folder = f"{ROOT}/{SPLIT}_series/{study}/{series}"
    keys = []
    for f in os.listdir(folder):
        h = pydicom.dcmread(f"{folder}/{f}", stop_before_pixels=True)
        keys.append((slice_key(h), f))
    keys.sort()
    n = len(keys)
    lo = int(0.2 * n); hi = max(int(0.8 * n), lo + 1)
    idx = np.clip(np.linspace(lo, hi - 1, K).round().astype(int), 0, n - 1)
    sl = []
    for i in idx:
        ds = pydicom.dcmread(f"{folder}/{keys[i][1]}")
        a = ds.pixel_array.astype(np.float32)
        if a.ndim == 3:
            a = a[..., 0] if a.shape[-1] in (3, 4) else a[0]
        a = a * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
        interp = cv2.INTER_AREA if a.shape[0] > S else cv2.INTER_LINEAR
        sl.append(cv2.resize(a, (S, S), interpolation=interp))
    v = np.stack(sl)
    p1, p99 = np.percentile(v, [1, 99])
    v = np.clip((v - p1) / max(p99 - p1, 1e-6), 0, 1)
    return (v * 255).astype(np.uint8), n

def process_study(job):
    study, series = job
    arr = np.zeros((3, K, S, S), np.uint8); info = {"StudyInstanceUID": study}
    for pi, s in enumerate(series):
        info[f"n_{PLANES[pi]}"] = 0; info[f"ok_{PLANES[pi]}"] = 0
        if s is None:
            continue
        try:
            arr[pi], n = load_series(study, s)
            info[f"n_{PLANES[pi]}"] = n; info[f"ok_{PLANES[pi]}"] = 1
        except Exception as e:
            info[f"err_{PLANES[pi]}"] = str(e)[:80]
    return arr, info

# **3: Model and weights**

 rebuild the exact architecture, with pretrained=False so nothing downloads, then load our trained weights.

In [ ]:
class KneeNet(nn.Module):
    def __init__(self, backbone, d, pool="mean"):
        super().__init__()
        self.bb, self.pool = backbone, pool
        self.att = nn.Linear(d, 1) if pool == "attn" else None
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(3 * d, 12))
    def forward(self, x):
        B, P, Ks, H, W = x.shape
        f = self.bb(x.reshape(B * P * Ks, 1, H, W)).view(B, P, Ks, -1)
        if self.pool == "mean":
            g = f.mean(2)
        else:
            wgt = torch.softmax(self.att(f).squeeze(-1), dim=2)
            g = (wgt.unsqueeze(-1) * f).sum(2)
        return self.head(g.reshape(B, -1))

model = KneeNet(timm.create_model("resnet18", pretrained=False, in_chans=1, num_classes=0), 512, "attn").to(dev)
wf = glob.glob("/kaggle/input/datasets/*/rsna-knee-weights/**/resnet18_attn.pt", recursive=True)[0]
model.load_state_dict(torch.load(wf, map_location=dev)); model.eval()
print("loaded", wf)

# **4: Inference and submission file**

process the test studies in chunks of 16 with 4 CPU workers, predict with the model, and write submission.csv. A failed study gets the prior instead of crashing the run.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
ts = pd.read_csv(f"{ROOT}/test_series.csv"); test = pd.read_csv(f"{ROOT}/test.csv")
cols = list(pd.read_csv(f"{ROOT}/sample_submission.csv").columns)
groups = {sid: g for sid, g in ts.groupby("StudyInstanceUID")}
jobs = [(sid, pick_series(groups[sid]) if sid in groups else [None, None, None]) for sid in test.StudyInstanceUID]

@torch.no_grad()
def infer(arrs):
    x = torch.from_numpy(np.stack(arrs)).to(dev).float().div_(255)
    x = (x - 0.45) / 0.225
    with torch.autocast("cuda", dtype=torch.float16, enabled=(dev == "cuda")):
        return torch.sigmoid(model(x).float()).cpu().numpy()

preds = np.tile(PRIOR, (len(jobs), 1)); CH = 16; t0 = time.time()
with ProcessPoolExecutor(max_workers=4) as ex:
    for s in range(0, len(jobs), CH):
        res = list(ex.map(process_study, jobs[s:s + CH]))
        ok = [i for i, (a, info) in enumerate(res) if sum(info[f"ok_{p}"] for p in PLANES) > 0]
        if ok:
            try:
                p = infer([res[i][0] for i in ok])
                for q, i in enumerate(ok): preds[s + i] = p[q]
            except Exception as e:
                print("batch failed, using prior:", str(e)[:100])
        print(f"{min(s + CH, len(jobs))}/{len(jobs)} studies | {time.time() - t0:.0f}s", flush=True)

sub = pd.DataFrame(preds, columns=LABELS); sub.insert(0, "StudyInstanceUID", test.StudyInstanceUID.values)
sub = sub[cols]
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.shape, "| NaNs:", int(sub.isna().sum().sum())); print(sub.head(3).round(3).to_string())